In [1]:
import os
import re

import pandas as pd
import matplotlib.pyplot as plt

from pyulog import ULog

In [2]:
# return the array of combined log file line
def getLogData(baseLogDir, dateTime, iteration, testCase, model="iris"):
	logPath = os.path.join(baseLogDir, dateTime, iteration, model, testCase, "log-combined.log_plain.log")
	log = None

	if os.path.exists(logPath):
		with open(logPath, "r") as f:
			log = f.readlines()

	return log

In [3]:
# return the ulg file name parsed from the combined log
def findUlgName(log):
	pattern = r"INFO\s+\[logger\]\s+Opened full log file:\s+(.*\.ulg)"
	ulg_file_name = ""

	match = re.search(pattern, log)

	if match:
		ulg_file_name = match.group(1)

	return ulg_file_name

In [4]:
# return the ulog parsed from the ulg file
def getUlogData(baseUlgDir, ulgFileName):
	ulog = None

	normPath = os.path.normpath(ulgFileName)
	ulgPath = os.path.join(baseUlgDir, normPath)

	if os.path.exists(ulgPath):
		ulog = ULog(ulgPath)

	return ulog

In [5]:
# ulg, combined log default location
baseLogDir = os.path.expanduser("~/PX4-Autopilot/logs")
baseUlgDir = os.path.expanduser("~/PX4-Autopilot/build/px4_sitl_default/tmp_mavsdk_tests/rootfs")

# combined log path data
experimentDateTime = "2025-03-10T17-34-40Z"
testCase = "bias_{x_0.03_y_0.00_z_0.00}_hold_20m"

# get the combined log data
combinedLog = getLogData(baseLogDir, experimentDateTime, "001", testCase)

# parse the ulg file name from the combined log
ulgFileName = findUlgName("".join(combinedLog))

# get the ulog data
ulog = getUlogData(baseUlgDir, ulgFileName)


# get the dataset from the ulog
vehicle_angular_velocity_groundtruth =  pd.DataFrame(ulog.get_dataset("vehicle_attitude_groundtruth").data)
vehicle_attitude_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_attitude_groundtruth").data)
vehicle_local_position_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_local_position_groundtruth").data)
vehicle_global_position_groundtruth = pd.DataFrame(ulog.get_dataset("vehicle_global_position_groundtruth").data)

print("vehicle_angular_velocity_groundtruth \n", vehicle_angular_velocity_groundtruth.head(1))
print("vehicle_attitude_groundtruth \n", vehicle_attitude_groundtruth.head(1))
print("vehicle_local_position_groundtruth \n", vehicle_local_position_groundtruth.head(1))
print("vehicle_global_position_groundtruth \n", vehicle_global_position_groundtruth.head(1))

vehicle_angular_velocity_groundtruth 
    timestamp  timestamp_sample     q[0]      q[1]      q[2]     q[3]  \
0   18496000                 0  0.70711  0.000393  0.000393  0.70711   

   delta_q_reset[0]  delta_q_reset[1]  delta_q_reset[2]  delta_q_reset[3]  \
0               0.0               0.0               0.0               0.0   

   quat_reset_counter  
0                   0  
vehicle_attitude_groundtruth 
    timestamp  timestamp_sample     q[0]      q[1]      q[2]     q[3]  \
0   18496000                 0  0.70711  0.000393  0.000393  0.70711   

   delta_q_reset[0]  delta_q_reset[1]  delta_q_reset[2]  delta_q_reset[3]  \
0               0.0               0.0               0.0               0.0   

   quat_reset_counter  
0                   0  
vehicle_local_position_groundtruth 
    timestamp  timestamp_sample  ref_timestamp    ref_lat   ref_lon    x    y  \
0   18496000                 0       17780000  47.397751  8.545607  0.0  0.0   

     z  delta_xy[0]  delta_xy[1]  ..

In [6]:
vehicle_angular_velocity_groundtruth['timestamp'] = pd.to_datetime(vehicle_angular_velocity_groundtruth['timestamp'], unit='us')
vehicle_angular_velocity_groundtruth.set_index("timestamp", inplace=True)
interpolated = vehicle_angular_velocity_groundtruth.interpolate(method="time")
interpolated

,timestamp_sample,q[0],q[1],q[2],q[3],delta_q_reset[0],delta_q_reset[1],delta_q_reset[2],delta_q_reset[3],quat_reset_counter
timestamp,,,,,,,,,,
1970-01-01 00:00:18.496,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.500,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.508,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.516,0,0.707110,0.000393,0.000393,0.707110,0.0,0.0,0.0,0.0,0
1970-01-01 00:00:18.524,0,0.707110,0.000393,0.000394,0.707110,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...
1970-01-01 00:02:25.636,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
1970-01-01 00:02:25.648,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
1970-01-01 00:02:25.656,0,0.000008,0.885512,0.464626,0.000101,0.0,0.0,0.0,0.0,0
